For each file, find its person id and split it into 5 second chunks

In [1]:
import os
import pandas as pd
import numpy as np
import time 

kalman_filter_data_path = '../data/kaggle-drdataboston/attempt_2/updated_data_kalman_filtered'
matrix_path = '../data/kaggle-drdataboston/matrix.csv'

out_path = '../data/kaggle-drdataboston/attempt_2'

X = []
y = []

window_ms = 5000
interp_pts = 500

matrix = pd.read_csv(matrix_path)

start_time = time.perf_counter()

for idx, file in enumerate(os.listdir(kalman_filter_data_path)):
    if idx % 5 == 0:
        c_time = time.perf_counter()
        print(f"Processed {idx} files in {c_time - start_time:.6f} seconds")

    if not file.endswith('.csv'):
        continue

    file_path = os.path.join(kalman_filter_data_path, file)
    df = pd.read_csv(file_path)


    clean_name = file
    if(clean_name.endswith('.csv.csv')):
        clean_name = clean_name[:-4]
    
    person_id = None
    for idx, row in matrix.iterrows():
        if clean_name.strip().lower() in [str(row.iloc[i]).strip().lower() for i in range(1, 5)]:
            person_id = row.iloc[0]
            break

    if person_id is None:
        continue

    df['window_id'] = (df['timestamp'] - df['timestamp'].iloc[0]) // window_ms

    chunks = []
    labels = []

    for _, group in df.groupby('window_id'):
        if len(group) < 2:
            continue

        time_arr = group['timestamp'].values 
        mag = group['filtered'].values
        time_arr = time_arr - time_arr[0]

        new_time = np.linspace(0, time_arr[-1], interp_pts)
        interp_mag = np.interp(new_time, time_arr, mag)

        chunks.append(interp_mag)
        labels.append(person_id)

    X.extend(chunks)
    y.extend(labels)


    print(np.array(X).shape)

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)
    

np.save(f"{out_path}/timewise_5s_500p_X.npy", X)
np.save(f"{out_path}/timewise_5s_500p_y.npy", y)



Processed 0 files in 0.000276 seconds
(81, 500)
(165, 500)
(259, 500)
(347, 500)
(452, 500)
Processed 5 files in 0.117530 seconds
(532, 500)
(633, 500)
(735, 500)
(828, 500)
(921, 500)
Processed 10 files in 0.247911 seconds
(1010, 500)
(1095, 500)
(1208, 500)
(1291, 500)
(1394, 500)
Processed 15 files in 0.361275 seconds
(1491, 500)
(1592, 500)
(1687, 500)
(1781, 500)
(1876, 500)
Processed 20 files in 0.478590 seconds
(1977, 500)
(2072, 500)
(2171, 500)
(2276, 500)
(2372, 500)
Processed 25 files in 0.607316 seconds
(2456, 500)
(2575, 500)
(2674, 500)
(2791, 500)
(2880, 500)
Processed 30 files in 0.733191 seconds
(2975, 500)
(3066, 500)
(3153, 500)
(3256, 500)
(3364, 500)
Processed 35 files in 0.853057 seconds
(3457, 500)
(3567, 500)
(3657, 500)
(3769, 500)
(3872, 500)
Processed 40 files in 0.978540 seconds
(3963, 500)
(4054, 500)
(4155, 500)
(4256, 500)
(4351, 500)
Processed 45 files in 1.098105 seconds
(4447, 500)
(4536, 500)
(4639, 500)
(4759, 500)
(4850, 500)
Processed 50 files in 1

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


2025-04-13 15:04:04.011695: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-13 15:04:04.020593: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-13 15:04:04.090678: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-13 15:04:04.151319: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744545844.204619   31812 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744545844.21

In [3]:
data_path = '../data/kaggle-drdataboston/attempt_2'
fname = 'timewise_5s_500p'

X = np.load(f'{data_path}/{fname}_X.npy')  # shape: (N, T)
y = np.load(f'{data_path}/{fname}_y.npy')  # shape: (N,)

print(X.shape, y.shape)


(26088, 500) (26088,)


In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder
# Fit on the full set of labels (before train/test split)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

joblib.dump(label_encoder, f'{data_path}/label_encoder.pkl')

# Then split the encoded labels
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [5]:
scaler = StandardScaler()

# Reshape for scaler: (N, T) → (N*T,)
X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
X_val_reshaped = X_val.reshape(X_val.shape[0], -1)
X_test_reshaped = X_test.reshape(X_test.shape[0], -1)

# Fit on training only
scaler.fit(X_train_reshaped)

joblib.dump(scaler, f'{data_path}/scaler.pkl')

X_train_scaled = scaler.transform(X_train_reshaped)
X_val_scaled = scaler.transform(X_val_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)


In [6]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_encoded))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),

    Dense(256, activation='relu'),
    Dropout(0.1),

    Dense(128, activation='relu'),

    Dense(num_classes, activation='softmax')
])


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_seq   = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
X_test_seq  = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, Input

model3 = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),  # (time_steps, 1)

    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


In [ ]:

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


model3.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=30,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop]
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, Input

model4 = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),  # (time_steps, 1)

    Conv1D(64, kernel_size=8, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(128, kernel_size=8, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


In [ ]:

model4.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


model4.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop]
)


In [8]:
from tensorflow.keras.models import load_model

model_lstm = load_model(f'{data_path}/model-cnn-lstm.keras')

2025-04-13 15:06:59.581381: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks = [early_stop]
)


In [ ]:
model2 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),

    Dense(num_classes, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


history = model2.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks = [early_stop]
)


In [9]:
test_loss, test_acc = model_lstm.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


123/123 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8219 - loss: 0.6289
Test accuracy: 0.83


In [ ]:
test_loss, test_acc = model2.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


In [ ]:
test_loss, test_acc = model3.evaluate(X_test_seq, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")

model3.save('../data/kaggle-drdataboston/attempt_2/model3-cnn.keras')


In [ ]:
test_loss, test_acc = model4.evaluate(X_test_seq, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")

model4.save('../data/kaggle-drdataboston/attempt_2/model4-cnn.keras')
